In [2]:
# import any required  packages here

import mat73
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import pandas as pd
import sklearn as sk
from statistics import mean
from sklearn.linear_model import LinearRegression
from scipy.signal import butter, lfilter, iirnotch

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

class SensorDataLoader:
    """Load and structure sensor data from Excel files to match HIR dataset format"""
    
    def __init__(self, base_folder):
        self.base_folder = Path(base_folder)
        
    def load_subject(self, subject_name):
        """Load all data for a single subject"""
        subject_folder = self.base_folder / f"{subject_name}_trials"
        
        # Skip katie_trials if needed
        if subject_name == "katie":
            return None
            
        # Load raw data and timestamps
        trial_file = subject_folder / "trial.xlsx"
        ts_file = subject_folder / "trial_timestamps.xlsx"
        
        raw = pd.read_excel(trial_file)
        ts = pd.read_excel(ts_file, header=None, names=['Time', 'Activity'])
        
        # Calculate sampling rate
        dt = raw['time'].diff().mean()
        sampling_rate = 1 / dt
        
        # Initialize subject structure
        subject_data = {}
        
        # Process each activity (timestamps come in start/end pairs)
        for i in range(0, len(ts), 2):
            if i+1 >= len(ts):
                break
                
            act_start = ts.loc[i, 'Time']
            act_end = ts.loc[i+1, 'Time']
            act_name = ts.loc[i, 'Activity']
            
            # Filter data for this activity
            mask = (raw['time'] >= act_start) & (raw['time'] <= act_end)
            activity_data = raw[mask].copy()
            
            if len(activity_data) == 0:
                continue
            
            # Create activity structure
            activity_struct = {}
            
            # Group columns by sensor type
            sensor_groups = self._identify_sensors(raw.columns)
            
            for sensor_name, pattern in sensor_groups.items():
                sensor_struct = self._extract_sensor_data(
                    activity_data, 
                    pattern, 
                    sampling_rate
                )
                if sensor_struct is not None:
                    activity_struct[sensor_name] = sensor_struct
            
            # Add to subject data
            subject_data[act_name] = activity_struct
        
        return subject_data
    
    def _identify_sensors(self, columns):
        """Identify sensor types from column names"""
        sensor_map = {}
        
        # IMU sensors - group all IMU data together (like APDM_Accel in original)
        imu_cols = [col for col in columns if any(x in col for x in [
            'AccelX_', 'AccelY_', 'AccelZ_',  # accelerometer
            'GyroX_', 'GyroY_', 'GyroZ_',     # gyroscope
            'MagX_', 'MagY_', 'MagZ_',        # magnetometer
            'Quat1_', 'Quat2_', 'Quat3_', 'Quat4_',  # quaternions
            'SensorIndex_', 'Sampletime_', 'Package_'  # metadata
        ])]
        
        if imu_cols:
            sensor_map['APDM_Accel'] = imu_cols
        
        # EMG sensors - if you have them
        emg_cols = [col for col in columns if 'emg' in col.lower() or any(
            muscle in col.lower() for muscle in ['glut', 'rect', 'vast', 'semi', 
            'bicfem', 'gastroc', 'sol', 'TA'])]
        if emg_cols:
            sensor_map['EMG'] = emg_cols
        
        # Wrist sensors - if you have them
        wrist_accel_cols = [col for col in columns if 'wrist' in col.lower() and 
                           any(x in col.lower() for x in ['accel', 'gyro'])]
        if wrist_accel_cols:
            sensor_map['Empatica_Accel'] = wrist_accel_cols
        
        # Physiological sensors - if you have them
        physio_cols = [col for col in columns if any(
            x in col.lower() for x in ['eda', 'temp', 'bvp', 'hr'])]
        if physio_cols:
            sensor_map['Empatica_Physio'] = physio_cols
        
        # Metabolic sensors - if you have them
        metabol_cols = [col for col in columns if any(
            x in col.lower() for x in ['vo2', 'vco2', 'rer', 'minvent', 'spo2'])]
        if metabol_cols:
            sensor_map['Metabolics_System'] = metabol_cols
        
        return sensor_map
    
    def _extract_sensor_data(self, df, columns, sampling_rate):
        """Extract data for a specific sensor"""
        if not columns or len(columns) == 0:
            return None
        
        # Get time and sensor data
        time_data = df['time'].values
        sensor_data = df[columns].values
        
        # Create code column (activity code - you may need to map activity names to codes)
        code = np.zeros((len(time_data), 1))
        
        # Combine: [time, code, sensor_data]
        data = np.column_stack([time_data, code, sensor_data])
        
        # Create labels
        labels = ['Time', 'Code'] + list(columns)
        
        return {
            'Labels': labels,
            'SamplingRate': sampling_rate,
            'Data': data
        }
    
    def load_all_subjects(self, subject_names):
        """Load all subjects into a dictionary"""
        all_subjects = {}
        
        for name in subject_names:
            print(f"Loading {name}...")
            data = self.load_subject(name)
            if data is not None:
                # Create cleaner subject key (e.g., "sam" -> "Subject_sam" or just use the name)
                subject_key = name.capitalize()
                all_subjects[subject_key] = data
        
        return all_subjects


# Helper function to get column indices for specific sensors/body parts
def get_sensor_indices(labels, sensor_location=None, sensor_type=None):
    """
    Get column indices for specific sensor data
    
    Args:
        labels: List of column labels from Data
        sensor_location: Body location (e.g., 'back', 'rshank', 'lthigh')
        sensor_type: Type of measurement ('Accel', 'Gyro', 'Mag', 'Quat')
    
    Returns:
        List of column indices
    """
    indices = []
    for i, label in enumerate(labels):
        match = True
        if sensor_location and sensor_location not in label:
            match = False
        if sensor_type and sensor_type not in label:
            match = False
        if match and label not in ['Time', 'Code']:
            indices.append(i)
    return indices


# Define body location constants for easier access (matching original code style)
# Indices for time and code
time_idx = 0
code_idx = 1

# For your data, define location-based indices
# Example: back sensor columns (you can create similar for each location)
def get_location_range(labels, location):
    """Get start and end indices for a body location"""
    indices = [i for i, label in enumerate(labels) if location in label]
    if indices:
        return [min(indices), max(indices) + 1]
    return [None, None]


# Example usage:
if __name__ == "__main__":
    # Set your base folder
    base_folder = r"\\iowa.uiowa.edu\shared\ResearchData\rdss_rvitali\Sam_Files\Research\NEEC\Pilot_Study"
    
    # Initialize loader
    loader = SensorDataLoader(base_folder)
    
    # Load Sam's data
    sam_data = loader.load_subject("sam")
    
    # Or use the load_all_subjects method (ready for when you add more subjects)
    all_data = loader.load_all_subjects(["sam"])
    
    # Access Sam's data:
    # Example: Get walking data
    # walking_data = all_data["Sam"]["Walking"]["APDM_Accel"]["Data"]
    # labels = all_data["Sam"]["Walking"]["APDM_Accel"]["Labels"]
    
    # Example: Get just the back sensor accelerometer data
    # back_accel_idx = get_sensor_indices(labels, 'back', 'Accel')
    # back_accel_data = walking_data[:, back_accel_idx]
    
    # Convert to DataFrame format matching your original structure
    result = pd.DataFrame(all_data)
    print(result)
    print("\nData structure:")
    if all_data:
        first_subject = list(all_data.keys())[0]
        print(f"\nSubject: {first_subject}")
        print(f"Activities: {list(all_data[first_subject].keys())}")
        if all_data[first_subject]:
            first_activity = list(all_data[first_subject].keys())[0]
            print(f"\nSensors in {first_activity}: {list(all_data[first_subject][first_activity].keys())}")
            if 'APDM_Accel' in all_data[first_subject][first_activity]:
                print(f"\nLabels (first 10): {all_data[first_subject][first_activity]['APDM_Accel']['Labels'][:10]}...")
                print(f"Data shape: {all_data[first_subject][first_activity]['APDM_Accel']['Data'].shape}")

Loading sam...
                                                      Sam
Jog     {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...
Fire    {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...
Wheel   {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...
Dummy   {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...
Stairs  {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...
Walk    {'APDM_Accel': {'Labels': ['Time', 'Code', 'Se...

Data structure:

Subject: Sam
Activities: ['Jog', 'Fire', 'Wheel', 'Dummy', 'Stairs', 'Walk']

Sensors in Jog: ['APDM_Accel']

Labels (first 10): ['Time', 'Code', 'SensorIndex_back', 'AccelX_back', 'AccelY_back', 'AccelZ_back', 'GyroX_back', 'GyroY_back', 'GyroZ_back', 'MagX_back']...
Data shape: (34499, 130)
